In [1]:
import pandas as pd
from pathlib import Path

CSV_PATH = (Path().resolve() / ".." / "data" / "raw" / "sample-sensor-data.csv").resolve()
df = pd.read_csv(CSV_PATH)

df.columns = [c.strip().lower() for c in df.columns]
df.head()



,timestamp,temperature,humidity,gas
0,2024-02-03 10:00:00,22.5,45.3,312
1,2024-02-03 10:00:01,22.6,45.5,315
2,2024-02-03 10:00:02,22.4,45.2,310
3,2024-02-03 10:00:03,22.7,45.8,318
4,2024-02-03 10:00:04,22.5,45.4,313


In [2]:
# Accept alternative names
col_map = {
    "temp": "temperature",
    "gas_level": "gas",
}
df = df.rename(columns=col_map)

needed = ["temperature","humidity","gas"]
for c in needed:
    if c not in df.columns:
        df[c] = 0

X = df[needed].fillna(0)
print(X.head())

   temperature  humidity  gas
0         22.5      45.3  312
1         22.6      45.5  315
2         22.4      45.2  310
3         22.7      45.8  318
4         22.5      45.4  313


In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
import joblib

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = IsolationForest(
    n_estimators=200,
    contamination=0.05,   # anomaly % (change as per your data)
    random_state=42
)
model.fit(X_scaled)

# -1 anomaly, 1 normal
pred = model.predict(X_scaled)
df["anomaly"] = (pred == -1)

# score (optional)
df["anomaly_score"] = -model.decision_function(X_scaled)

# save outputs
out_csv = (Path().resolve() / ".." / "data" / "processed" / "labeled_anomaly.csv").resolve()
out_csv.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out_csv, index=False)

# save artifacts to backend/models
models_dir = (Path().resolve() / ".." / "backend" / "models").resolve()
models_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(model, models_dir / "isolation_forest.joblib")
joblib.dump(scaler, models_dir / "scaler.joblib")

print("Saved:", out_csv)
print("Saved model:", models_dir / "isolation_forest.joblib")


Saved: D:\internship project\SelfHealing-IoT-AI\backend\data\processed\labeled_anomaly.csv
Saved model: D:\internship project\SelfHealing-IoT-AI\backend\backend\models\isolation_forest.joblib


In [4]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# true labels (already in CSV)
y_true = df["anomaly"].astype(int)

# model prediction
y_pred = (pred == -1).astype(int)

accuracy = accuracy_score(y_true, y_pred)

print("Accuracy:", round(accuracy * 100, 2), "%")



Accuracy: 100.0 %
